# FedAC — matched full-participation baseline

FedAC local/server operations with LatentFed's matched partitions, holdout, initialization and 20/20-client budget. The common protocol is specified in docs/EXPERIMENT_PROTOCOL.md.

Readable implementation: src/latentfed_repro. Imports never start training. Both datasets are enabled. Smoke outputs are synthetic and cannot enter paper statistics.

## 1. Setup
Open from the complete checkout. A standalone Kaggle notebook can clone the published repository. Enable Internet and T4. Keep the compatible platform torch/torchvision pair.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, importlib.metadata

REPOSITORY_URL = "https://github.com/vialt8/LatentFed.git"
candidates = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working/LatentFed")]
ROOT = next((p for p in candidates if (p / "src/latentfed_repro").is_dir()), None)
if ROOT is None:
    destination = Path("/kaggle/working/LatentFed") if Path("/kaggle/working").is_dir() else Path.cwd() / "LatentFed"
    if not destination.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPOSITORY_URL, str(destination)])
    ROOT = destination
    if not (ROOT / "src/latentfed_repro").is_dir():
        raise FileNotFoundError("Upload this release to GitHub first, or open this notebook inside the complete supplied folder.")
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
for name in ("torch", "torchvision"):
    if importlib.util.find_spec(name) is None:
        raise RuntimeError("Install a compatible torch/torchvision pair as documented in README, then restart the kernel.")
requirements = [
    ("numpy", "numpy", "2.1.3"), ("pandas", "pandas", "2.3.3"),
    ("scipy", "scipy", "1.16.3"), ("sklearn", "scikit-learn", "1.6.1"),
    ("hdbscan", "hdbscan", "0.8.44")]
install, loaded = [], []
for module, distribution, version in requirements:
    try:
        current = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != version:
        install.append(f"{distribution}=={version}")
        if module in sys.modules:
            loaded.append(module)
for module, requirement in [("matplotlib", "matplotlib>=3.9,<3.11"), ("tqdm", "tqdm>=4.66,<5")]:
    if importlib.util.find_spec(module) is None:
        install.append(requirement)
if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install])
if loaded:
    raise RuntimeError("Dependencies were updated after being imported. Restart the kernel and Run All to use the installed versions: " + ", ".join(loaded))
print("Source checkout:", ROOT)


## 2. Select the batch
Defaults specify the complete paper grid. A shorter real-data batch can select one official seed and alpha while retaining both datasets. Use the same selection across methods.

In [ ]:
from latentfed_repro.config import experiment_plan
from latentfed_repro.experiments import run_grid
from IPython.display import display
import pandas as pd

DATASETS = ["cifar10", "emnist"]
ALPHAS = [0.3, 0.5, 0.7]
SEEDS = [42, 101, 2024, 3407, 8888]
VARIANTS = ['full']
METHOD = 'fedac'
SMOKE_TEST = os.environ.get("LATENTFED_SMOKE", "0") == "1"  # set True for synthetic checks
default_output = Path("/kaggle/working/latentfed_repro_outputs") if Path("/kaggle/working").is_dir() else ROOT / "outputs"
OUTPUT_ROOT = Path(os.environ.get("LATENTFED_OUTPUT_ROOT", str(default_output)))
RESUME = True
INCLUDE_CHECKPOINTS_IN_ZIP = False

PLAN = experiment_plan(DATASETS, ALPHAS, SEEDS, VARIANTS, smoke=SMOKE_TEST)
print("MODE:", "SMOKE: synthetic, not paper results" if SMOKE_TEST else "PAPER: real datasets, full training")
print("Experiments in this invocation:", len(PLAN))
print("Clients: 20; selected per round: 20; local epochs: 1; batch size: 32")
display(pd.DataFrame(PLAN).head(12))


## 3. Inspect the model
The baseline uses the common classifier, initialization and 20-client participation schedule. Its method-specific updates are implemented in the source module.

In [ ]:
from latentfed_repro.config import make_config
from latentfed_repro import models
import torch

architecture_rows = []
for variant in VARIANTS:
    cfg = make_config(DATASETS[0], ALPHAS[0], SEEDS[0], variant, method=METHOD, smoke=SMOKE_TEST)
    channels, size, classes = (3, 32, 10) if DATASETS[0] == "cifar10" else (1, 28, 47)
    if METHOD == "latentfed":
        model = models.make_autoencoder(channels, size, cfg)
        architecture_rows.append(dict(variant=variant, autoencoder=type(model).__name__,
            MDFE=cfg.mdfe_mode != "off", CALD=cfg.cald_mode != "off",
            parameters=sum(p.numel() for p in model.parameters()),
            convolution_layers=sum(isinstance(m, (torch.nn.Conv2d, torch.nn.ConvTranspose2d)) for m in model.modules())))
    else:
        model = models.SimpleCNN(channels, classes)
        architecture_rows.append(dict(method=METHOD, classifier=type(model).__name__,
            clients_per_round=20, auxiliary_autoencoder=False,
            parameters=sum(p.numel() for p in model.parameters())))
display(pd.DataFrame(architecture_rows))
if METHOD == "latentfed" and "ae_only" in VARIANTS:
    print(models.FullyConnectedAutoencoder(channels, size, latent_dim=128, hidden_dim=512))


## 4. Run
CIFAR-10 uses a pinned, checksum-verified Hugging Face archive; EMNIST is the 47-class Balanced split with orientation correction. No Add Input or HF token is needed. Paper mode uses 100 rounds and CUDA by default. Smoke mode uses synthetic images, seed 7 and two rounds with 20/20 clients; CALD warm-up is shortened only for this diagnostic mode.

In [ ]:
RESULT = run_grid(method=METHOD, datasets=DATASETS, alphas=ALPHAS, seeds=SEEDS,
    variants=VARIANTS, output=OUTPUT_ROOT, smoke=SMOKE_TEST, resume=RESUME,
    include_checkpoints=INCLUDE_CHECKPOINTS_IN_ZIP)
print("Completed run folders:", len(RESULT["runs"]))
print("ZIP:", RESULT["archive"])


## 5. Results and coverage
Each paper endpoint averages rounds 91–100. Publication rows require five distinct official seeds and sample SD (ddof=1) across endpoints. Undefined metrics remain missing. FedProto uses nearest global prototypes for primary accuracy, and has no cluster metrics.

In [ ]:
import json
status = []
for folder in RESULT["runs"]:
    manifest = json.loads((folder / "manifest.json").read_text())
    status.append({k: manifest[k] for k in ["method", "variant", "dataset", "alpha", "seed", "mode", "completed_rounds", "publication_eligible"]})
display(pd.DataFrame(status))
if RESULT["summary"].empty:
    print("No complete five-seed publication group. Smoke/partial groups are not substituted.")
else:
    display(RESULT["summary"])
from latentfed_repro.plots import plot_run_preview
import matplotlib.pyplot as plt
figure = plot_run_preview(RESULT["runs"])
display(figure)
plt.close(figure)


## 6. Download and resume
The final cell creates the ZIP automatically. It is also refreshed after each experiment. Round/client metrics, diagnostics, manifests and coverage are included. Each run has a local checkpoint. Enable INCLUDE_CHECKPOINTS_IN_ZIP or download checkpoints separately before a Kaggle session ends. Restore the same folder hierarchy, source and scientific settings to resume. Completed runs are reused, not additional seeds.

In [ ]:
from latentfed_repro.artifacts import export_zip
from IPython.display import FileLink, display
MODE_ROOT = OUTPUT_ROOT / ("smoke" if SMOKE_TEST else "paper") / METHOD
DOWNLOAD_ZIP = export_zip(MODE_ROOT, include_checkpoints=INCLUDE_CHECKPOINTS_IN_ZIP)
link_path = str(DOWNLOAD_ZIP) if Path("/kaggle/working").is_dir() else os.path.relpath(DOWNLOAD_ZIP, Path.cwd())
display(FileLink(link_path))
print("Download from this link or Kaggle Output:", DOWNLOAD_ZIP)
